# Prepare dataset for classification training

## Imports

In [1]:
import pandas as pd
import numpy as np

## Importing the Music Dataset

In [2]:
music = pd.read_csv("music.csv")

In [3]:
music.head()

,title,rank,date,artist,url,region,chart,trend,streams
0,Chantaje (feat. Maluma),1,2017-01-01,Shakira,https://open.spotify.com/track/6mICuAdrwEjh6Y6...,Argentina,top200,SAME_POSITION,253019.0
1,Vente Pa' Ca (feat. Maluma),2,2017-01-01,Ricky Martin,https://open.spotify.com/track/7DM4BPaS7uofFul...,Argentina,top200,MOVE_UP,223988.0
2,Reggaetón Lento (Bailemos),3,2017-01-01,CNCO,https://open.spotify.com/track/3AEZUABDXNtecAO...,Argentina,top200,MOVE_DOWN,210943.0
3,Safari,4,2017-01-01,"J Balvin, Pharrell Williams, BIA, Sky",https://open.spotify.com/track/6rQSrBHf7HlZjtc...,Argentina,top200,SAME_POSITION,173865.0
4,Shaky Shaky,5,2017-01-01,Daddy Yankee,https://open.spotify.com/track/58IL315gMSTD37D...,Argentina,top200,MOVE_UP,153956.0


In [4]:
music[music["streams"] > 500_000].shape[0]

444644

## Merging Primary Genre Labels

In [5]:
genres = pd.read_csv("artists_merged_clean_NAN_to_pop.csv")

In [6]:
genres.head()

,artist,genre
0,Shakira,"pop, latin, female vocalists, spanish, rock, d..."
1,Ricky Martin,"pop, latin, latin pop, dance, spanish, male vo..."
2,CNCO,"pop, latin, Reggaeton, latin pop, boybands, ma..."
3,"J Balvin, Pharrell Williams, BIA, Sky",pop
4,Daddy Yankee,"Reggaeton, latin, Hip-Hop, spanish, rap, puert..."


In [7]:
genres["main_genre"] = (
    genres["genre"]
    .str.split(",")
    .str[0]
    .str.strip()
    .str.lower()
)

music = music.merge(
    genres[["artist", "main_genre"]],
    on="artist",
    how="left"
)

In [8]:
music.head()

,title,rank,date,artist,url,region,chart,trend,streams,main_genre
0,Chantaje (feat. Maluma),1,2017-01-01,Shakira,https://open.spotify.com/track/6mICuAdrwEjh6Y6...,Argentina,top200,SAME_POSITION,253019.0,pop
1,Vente Pa' Ca (feat. Maluma),2,2017-01-01,Ricky Martin,https://open.spotify.com/track/7DM4BPaS7uofFul...,Argentina,top200,MOVE_UP,223988.0,pop
2,Reggaetón Lento (Bailemos),3,2017-01-01,CNCO,https://open.spotify.com/track/3AEZUABDXNtecAO...,Argentina,top200,MOVE_DOWN,210943.0,pop
3,Safari,4,2017-01-01,"J Balvin, Pharrell Williams, BIA, Sky",https://open.spotify.com/track/6rQSrBHf7HlZjtc...,Argentina,top200,SAME_POSITION,173865.0,pop
4,Shaky Shaky,5,2017-01-01,Daddy Yankee,https://open.spotify.com/track/58IL315gMSTD37D...,Argentina,top200,MOVE_UP,153956.0,reggaeton


## Dataset Overview and Missing Values

In [9]:
music.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26173485 entries, 0 to 26173484
Data columns (total 10 columns):
 #   Column      Dtype  
---  ------      -----  
 0   title       object 
 1   rank        int64  
 2   date        object 
 3   artist      object 
 4   url         object 
 5   region      object 
 6   chart       object 
 7   trend       object 
 8   streams     float64
 9   main_genre  object 
dtypes: float64(1), int64(1), object(8)
memory usage: 2.0+ GB


In [10]:
music.describe()

,rank,streams
count,2.617348e+07,2.617348e+07
mean,8.092316e+01,4.504503e+04
std,5.918601e+01,1.856572e+05
min,1.000000e+00,1.001000e+03
25%,2.900000e+01,4.850000e+03
50%,6.700000e+01,9.565000e+03
75%,1.310000e+02,2.489900e+04
max,2.000000e+02,1.974970e+07


In [11]:
music.isnull().sum()

title          0
rank           0
date           0
artist         0
url            0
region         0
chart          0
trend          0
streams        0
main_genre    69
dtype: int64

In [12]:
music['main_genre'] = music['main_genre'].fillna('pop')

In [13]:
music.isnull().sum()

title         0
rank          0
date          0
artist        0
url           0
region        0
chart         0
trend         0
streams       0
main_genre    0
dtype: int64

## Preprocessing

In [14]:
music = music.drop(columns=["url"])
music = music.drop(columns=["title"])
music = music.drop(columns=["chart"])
music.head()

,rank,date,artist,region,trend,streams,main_genre
0,1,2017-01-01,Shakira,Argentina,SAME_POSITION,253019.0,pop
1,2,2017-01-01,Ricky Martin,Argentina,MOVE_UP,223988.0,pop
2,3,2017-01-01,CNCO,Argentina,MOVE_DOWN,210943.0,pop
3,4,2017-01-01,"J Balvin, Pharrell Williams, BIA, Sky",Argentina,SAME_POSITION,173865.0,pop
4,5,2017-01-01,Daddy Yankee,Argentina,MOVE_UP,153956.0,reggaeton


## Creating the Hit Label

In [15]:
music["is_hit"] = (music["streams"] > 500_000).astype("int8")
music.head()

,rank,date,artist,region,trend,streams,main_genre,is_hit
0,1,2017-01-01,Shakira,Argentina,SAME_POSITION,253019.0,pop,0
1,2,2017-01-01,Ricky Martin,Argentina,MOVE_UP,223988.0,pop,0
2,3,2017-01-01,CNCO,Argentina,MOVE_DOWN,210943.0,pop,0
3,4,2017-01-01,"J Balvin, Pharrell Williams, BIA, Sky",Argentina,SAME_POSITION,173865.0,pop,0
4,5,2017-01-01,Daddy Yankee,Argentina,MOVE_UP,153956.0,reggaeton,0


### Creating Date Features

In [16]:
music["date"] = pd.to_datetime(music["date"])
music["year"] = music["date"].dt.year.astype("int16")
music["month"] = music["date"].dt.month.astype("int8")
music["day"] = music["date"].dt.day.astype("int8")
music["weekday"] = music["date"].dt.weekday.astype("int8")
music["quarter"] = music["date"].dt.quarter.astype("int8")
music["is_weekend"] = music["weekday"].isin([5, 6]).astype("int8")
music["days_since_start"] = (music["date"] - music["date"].min()).dt.days.astype("int32")
music = music.drop(columns=["date"])
music.head()

,rank,artist,region,trend,streams,main_genre,is_hit,year,month,day,weekday,quarter,is_weekend,days_since_start
0,1,Shakira,Argentina,SAME_POSITION,253019.0,pop,0,2017,1,1,6,1,1,0
1,2,Ricky Martin,Argentina,MOVE_UP,223988.0,pop,0,2017,1,1,6,1,1,0
2,3,CNCO,Argentina,MOVE_DOWN,210943.0,pop,0,2017,1,1,6,1,1,0
3,4,"J Balvin, Pharrell Williams, BIA, Sky",Argentina,SAME_POSITION,173865.0,pop,0,2017,1,1,6,1,1,0
4,5,Daddy Yankee,Argentina,MOVE_UP,153956.0,reggaeton,0,2017,1,1,6,1,1,0


### Encoding the Region and Trend Columns

In [17]:
music = pd.get_dummies(music, columns=["trend"], sparse=False)

bool_cols = music.select_dtypes(include="bool").columns
music[bool_cols] = music[bool_cols].astype("int8")
music.head()

,rank,artist,region,streams,main_genre,is_hit,year,month,day,weekday,quarter,is_weekend,days_since_start,trend_MOVE_DOWN,trend_MOVE_UP,trend_NEW_ENTRY,trend_SAME_POSITION
0,1,Shakira,Argentina,253019.0,pop,0,2017,1,1,6,1,1,0,0,0,0,1
1,2,Ricky Martin,Argentina,223988.0,pop,0,2017,1,1,6,1,1,0,0,1,0,0
2,3,CNCO,Argentina,210943.0,pop,0,2017,1,1,6,1,1,0,1,0,0,0
3,4,"J Balvin, Pharrell Williams, BIA, Sky",Argentina,173865.0,pop,0,2017,1,1,6,1,1,0,0,0,0,1
4,5,Daddy Yankee,Argentina,153956.0,reggaeton,0,2017,1,1,6,1,1,0,0,1,0,0


### Artist Target Encoding

In [18]:
artist_count = music["artist"].value_counts()
music["artist_count"] = music["artist"].map(artist_count)

In [19]:
music.describe()

,rank,streams,is_hit,year,month,day,weekday,quarter,is_weekend,days_since_start,trend_MOVE_DOWN,trend_MOVE_UP,trend_NEW_ENTRY,trend_SAME_POSITION,artist_count
count,2.617348e+07,2.617348e+07,2.617348e+07,2.617348e+07,2.617348e+07,2.617348e+07,2.617348e+07,2.617348e+07,2.617348e+07,2.617348e+07,2.617348e+07,2.617348e+07,2.617348e+07,2.617348e+07,2.617348e+07
mean,8.092316e+01,4.504503e+04,1.698834e-02,2.019135e+03,6.523776e+00,1.574077e+01,2.997616e+00,2.511580e+00,2.843784e-01,9.616494e+02,4.286941e-01,3.744646e-01,7.082118e-02,1.260201e-01,3.713276e+04
std,5.918601e+01,1.856572e+05,1.292275e-01,1.390380e+00,3.413495e+00,8.792205e+00,1.997265e+00,1.111316e+00,4.511179e-01,5.147829e+02,4.948894e-01,4.839844e-01,2.565259e-01,3.318721e-01,6.773955e+04
min,1.000000e+00,1.001000e+03,0.000000e+00,2.017000e+03,1.000000e+00,1.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
25%,2.900000e+01,4.850000e+03,0.000000e+00,2.018000e+03,4.000000e+00,8.000000e+00,1.000000e+00,2.000000e+00,0.000000e+00,5.310000e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.583000e+03
50%,6.700000e+01,9.565000e+03,0.000000e+00,2.019000e+03,7.000000e+00,1.600000e+01,3.000000e+00,3.000000e+00,0.000000e+00,9.900000e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,8.139000e+03
75%,1.310000e+02,2.489900e+04,0.000000e+00,2.020000e+03,9.000000e+00,2.300000e+01,5.000000e+00,3.000000e+00,1.000000e+00,1.410000e+03,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,3.817400e+04
max,2.000000e+02,1.974970e+07,1.000000e+00,2.021000e+03,1.200000e+01,3.100000e+01,6.000000e+00,4.000000e+00,1.000000e+00,1.825000e+03,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,3.879170e+05


## Load preprocesses csv

In [20]:
music.to_csv('music_classification.csv', index=False)